# FlowRegime One — Colab A100/L4 GPU Quantitative Regime & Volatility Training Notebook

**Upswing-Labs | AKINDO WaveHack Wave 3: One-Person On-Chain Finance with SoSoValue**

This 1-click Google Colab GPU notebook executes the complete institutional quantitative research pipeline behind **FlowRegime One**:

1. **Real Data Ingestion**: Fetches official US Spot BTC/ETH ETF inflows from **SoSoValue OpenAPI v1** (`GET /openapi/v1/etfs/summary-history`, header `x-soso-api-key`) when configured, with automatic live public fallback (Kraken 720-day BTC/ETH/SOL OHLCV + DefiLlama Global Stablecoin Flows + Alternative.me Fear & Greed).
2. **PyTorch TCN + Multi-Head Attention (`FlowRegimeTCNAttn`)**: Causal dilated Temporal Convolutional Network + Multi-Head Self-Attention predicting 4 market regimes (`ACCUMULATION_BULL`, `ETF_MOMENTUM_EXPANSION`, `HIGH_VOL_CHOP`, `DISTRIBUTION_DRAWDOWN`), 24h expected return, and 72h annualized realized volatility.
3. **Strict Purged Walk-Forward CV**: 5-fold expanding walk-forward evaluation with a **10-day embargo gap** (`>= 7-day label horizon`) guaranteeing **zero lookahead leakage**.
4. **Knowledge Distillation to `$0`-GPU Edge Weights**: Distills the trained GPU TCN-Attention ensemble into a compact `12 -> 16 (tanh) -> 6` edge MLP exported as `model_weights.json` (runs in `<1ms` inside Cloudflare Workers with `$0` runtime GPU cost!) plus verifiable `walkforward_metrics.json`.

> **Runtime Tip**: In Google Colab, select **Runtime -> Change runtime type -> A100 or L4 GPU** (all required libraries — `torch`, `numpy`, `pandas`, `requests`, `matplotlib` — are preinstalled on Colab).

In [ ]:
# Cell 1: Hardware Detection, AMP Precision Setup & Deterministic Seeding
import json
import math
import os
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (
    torch.bfloat16
    if (torch.cuda.is_available() and torch.cuda.is_bf16_supported())
    else torch.float16
)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU fallback"
print(f"FlowRegime One Quant Lab | Device: {DEVICE} ({gpu_name}) | AMP dtype: {AMP_DTYPE}")

In [ ]:
# Cell 2: Real Historical Market & SoSoValue OpenAPI v1 Data Loader
SOSOVALUE_BASE_URL = "https://openapi.sosovalue.com/openapi/v1"
REGIMES = [
    "ACCUMULATION_BULL",
    "ETF_MOMENTUM_EXPANSION",
    "HIGH_VOL_CHOP",
    "DISTRIBUTION_DRAWDOWN",
]
FEATURE_NAMES = [
    "btc_ret_1d",
    "btc_ret_7d",
    "btc_ret_30d",
    "eth_btc_rel_7d",
    "sol_btc_rel_7d",
    "realized_vol_7d",
    "realized_vol_30d",
    "vol_ratio_7_30",
    "flow_zscore_7d",
    "flow_zscore_30d",
    "drawdown_30d",
    "sentiment_norm",
]


def resolve_sosovalue_api_key() -> str:
    env_key = os.environ.get("SOSOVALUE_API_KEY", "").strip()
    if env_key:
        return env_key
    try:
        from google.colab import userdata  # type: ignore

        return (userdata.get("SOSOVALUE_API_KEY") or "").strip()
    except Exception:
        return ""


def fetch_sosovalue_etf_history(symbol: str, api_key: str, limit: int = 365) -> dict:
    if not api_key:
        return {}
    url = f"{SOSOVALUE_BASE_URL}/etfs/summary-history"
    resp = requests.get(
        url,
        params={"symbol": symbol, "country_code": "US", "limit": limit},
        headers={"x-soso-api-key": api_key, "User-Agent": "FlowRegimeOne-Colab/1.0"},
        timeout=20,
    )
    if resp.status_code != 200:
        print(f"[SoSoValue] {symbol} ETF endpoint returned HTTP {resp.status_code}; using public liquidity proxy.")
        return {}
    payload = resp.json()
    if payload.get("code") != 0 or not isinstance(payload.get("data"), list):
        return {}
    out = {}
    for row in payload["data"]:
        # Respect T+1 settlement: skip unsettled intraday null flow rows
        if row.get("date") and row.get("total_net_inflow") is not None:
            out[str(row["date"])] = float(row["total_net_inflow"])
    print(f"[SoSoValue] Loaded {len(out)} settled US Spot {symbol} ETF daily flow records.")
    return out


def fetch_kraken_daily(pair: str) -> pd.DataFrame:
    url = f"https://api.kraken.com/0/public/OHLC?pair={pair}&interval=1440"
    payload = requests.get(url, headers={"User-Agent": "FlowRegimeOne-Colab/1.0"}, timeout=20).json()
    res = payload.get("result", {})
    series_key = next(k for k in res.keys() if k != "last")
    rows = []
    for r in res[series_key]:
        day = datetime.fromtimestamp(int(r[0]), tz=timezone.utc).strftime("%Y-%m-%d")
        rows.append(
            {
                "date": day,
                "open": float(r[1]),
                "high": float(r[2]),
                "low": float(r[3]),
                "close": float(r[4]),
                "vwap": float(r[5]),
                "volume_usd": float(r[6]) * float(r[5]),
            }
        )
    return pd.DataFrame(rows).drop_duplicates(subset=["date"]).sort_values("date").set_index("date")


def fetch_defillama_stables() -> pd.Series:
    payload = requests.get(
        "https://stablecoins.llama.fi/stablecoincharts/all",
        headers={"User-Agent": "FlowRegimeOne-Colab/1.0"},
        timeout=20,
    ).json()
    data = {}
    for row in payload:
        ts = int(row.get("date", 0))
        val = float((row.get("totalCirculatingUSD") or {}).get("peggedUSD") or 0.0)
        if ts > 0 and val > 0:
            day = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d")
            data[day] = val
    return pd.Series(data, name="stable_usd").sort_index()


def fetch_fear_greed() -> pd.Series:
    payload = requests.get(
        "https://api.alternative.me/fng/?limit=750",
        headers={"User-Agent": "FlowRegimeOne-Colab/1.0"},
        timeout=20,
    ).json()
    data = {}
    for row in payload.get("data", []):
        ts = int(row.get("timestamp", 0))
        if ts > 0:
            day = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m-%d")
            data[day] = float(row.get("value", 50.0))
    return pd.Series(data, name="fng").sort_index()


soso_key = resolve_sosovalue_api_key()
soso_btc_etf = fetch_sosovalue_etf_history("BTC", soso_key)

btc_df = fetch_kraken_daily("XBTUSD")
eth_df = fetch_kraken_daily("ETHUSD")
sol_df = fetch_kraken_daily("SOLUSD")
stables_s = fetch_defillama_stables()
fng_s = fetch_fear_greed()

common_idx = btc_df.index.intersection(eth_df.index).intersection(sol_df.index).sort_values()
df = pd.DataFrame(index=common_idx)
df["btc_close"] = btc_df.loc[df.index, "close"]
df["btc_high"] = btc_df.loc[df.index, "high"]
df["btc_vol_usd"] = btc_df.loc[df.index, "volume_usd"]
df["eth_close"] = eth_df.loc[df.index, "close"]
df["sol_close"] = sol_df.loc[df.index, "close"]
df["stable_usd"] = stables_s.reindex(df.index).ffill().bfill()
df["fng"] = fng_s.reindex(df.index).ffill().bfill().fillna(50.0)
df["soso_etf_flow"] = pd.Series(soso_btc_etf).reindex(df.index)

USING_OFFICIAL_SOSO_ETF = int(df["soso_etf_flow"].notna().sum()) >= 60
print(
    f"Aligned market history: {len(df)} daily bars ({df.index[0]} -> {df.index[-1]}) "
    f"| Official SoSoValue ETF flows active: {USING_OFFICIAL_SOSO_ETF}"
)

In [ ]:
# Cell 3: Strictly Non-Leaking 12-Factor Feature Engineering & 7-Day Forward Regime Targets
HORIZON_DAYS = 7
SEQ_LEN = 21

df["btc_ret_1d"] = np.log(df["btc_close"] / df["btc_close"].shift(1))
df["eth_ret_1d"] = np.log(df["eth_close"] / df["eth_close"].shift(1))
df["sol_ret_1d"] = np.log(df["sol_close"] / df["sol_close"].shift(1))

df["btc_ret_7d"] = np.log(df["btc_close"] / df["btc_close"].shift(7))
df["btc_ret_30d"] = np.log(df["btc_close"] / df["btc_close"].shift(30))
df["eth_btc_rel_7d"] = np.log(df["eth_close"] / df["eth_close"].shift(7)) - df["btc_ret_7d"]
df["sol_btc_rel_7d"] = np.log(df["sol_close"] / df["sol_close"].shift(7)) - df["btc_ret_7d"]

df["realized_vol_7d"] = df["btc_ret_1d"].rolling(7).std() * np.sqrt(365.0)
df["realized_vol_30d"] = df["btc_ret_1d"].rolling(30).std() * np.sqrt(365.0)
df["vol_ratio_7_30"] = df["realized_vol_7d"] / df["realized_vol_30d"].clip(lower=0.08)

# Official SoSoValue US Spot BTC ETF net inflow when available, else DefiLlama + signed volume proxy
proxy_flow = df["stable_usd"].diff().fillna(0.0) + np.tanh(df["btc_ret_1d"].fillna(0.0) * 18.0) * df["btc_vol_usd"] * 0.12
df["net_flow_usd"] = df["soso_etf_flow"].fillna(proxy_flow) if USING_OFFICIAL_SOSO_ETF else proxy_flow

flow_m30 = df["net_flow_usd"].rolling(30).mean()
flow_s30 = df["net_flow_usd"].rolling(30).std().clip(lower=1e7)
flow_m7 = df["net_flow_usd"].rolling(7).mean()
df["flow_zscore_7d"] = ((flow_m7 - flow_m30) / flow_s30).clip(-3.5, 3.5)
df["flow_zscore_30d"] = (flow_m30 / flow_s30).clip(-3.5, 3.5)

rolling_peak_30 = df["btc_high"].rolling(30).max()
df["drawdown_30d"] = (df["btc_close"] - rolling_peak_30) / rolling_peak_30.clip(lower=1.0)
df["sentiment_norm"] = ((df["fng"] - 50.0) / 50.0).clip(-1.0, 1.0)

# Strictly FORWARD targets (t+1 .. t+HORIZON_DAYS) — NEVER mixed into feature tensor!
df["fwd_1d"] = np.log(df["btc_close"].shift(-1) / df["btc_close"])
df["fwd_eth_1d"] = np.log(df["eth_close"].shift(-1) / df["eth_close"])
df["fwd_sol_1d"] = np.log(df["sol_close"].shift(-1) / df["sol_close"])
df["fwd_7d"] = np.log(df["btc_close"].shift(-HORIZON_DAYS) / df["btc_close"])
df["fwd_vol_7d"] = (
    df["btc_ret_1d"].shift(-1).rolling(HORIZON_DAYS).std().shift(-(HORIZON_DAYS - 1)) * np.sqrt(365.0)
).clip(0.15, 1.60)


def assign_regime_label(row) -> int:
    if row["fwd_7d"] >= 0.035:
        return 1  # ETF_MOMENTUM_EXPANSION
    if row["fwd_7d"] <= -0.035:
        return 3  # DISTRIBUTION_DRAWDOWN
    if row["fwd_vol_7d"] >= 0.44:
        return 2  # HIGH_VOL_CHOP
    return 0      # ACCUMULATION_BULL


clean_df = df.dropna(subset=FEATURE_NAMES + ["fwd_1d", "fwd_7d", "fwd_vol_7d"]).copy()
clean_df["label"] = clean_df.apply(assign_regime_label, axis=1).astype(int)

dates_arr = clean_df.index.to_numpy()
feat_mat = clean_df[FEATURE_NAMES].to_numpy(dtype=np.float32)
labels_arr = clean_df["label"].to_numpy(dtype=np.int64)
fwd1_arr = clean_df["fwd_1d"].to_numpy(dtype=np.float32)
fwd_eth_arr = clean_df["fwd_eth_1d"].to_numpy(dtype=np.float32)
fwd_sol_arr = clean_df["fwd_sol_1d"].to_numpy(dtype=np.float32)
fwd_vol_arr = clean_df["fwd_vol_7d"].to_numpy(dtype=np.float32)

seq_X, cur_X, seq_y, seq_r1, seq_re1, seq_rs1, seq_vol, seq_dates = [], [], [], [], [], [], [], []
for idx in range(SEQ_LEN - 1, len(clean_df)):
    seq_X.append(feat_mat[idx - SEQ_LEN + 1 : idx + 1])
    cur_X.append(feat_mat[idx])
    seq_y.append(labels_arr[idx])
    seq_r1.append(fwd1_arr[idx])
    seq_re1.append(fwd_eth_arr[idx])
    seq_rs1.append(fwd_sol_arr[idx])
    seq_vol.append(fwd_vol_arr[idx])
    seq_dates.append(str(dates_arr[idx]))

seq_X = np.asarray(seq_X, dtype=np.float32)
cur_X = np.asarray(cur_X, dtype=np.float32)
seq_y = np.asarray(seq_y, dtype=np.int64)
seq_r1 = np.asarray(seq_r1, dtype=np.float32)
seq_re1 = np.asarray(seq_re1, dtype=np.float32)
seq_rs1 = np.asarray(seq_rs1, dtype=np.float32)
seq_vol = np.asarray(seq_vol, dtype=np.float32)

counts = {REGIMES[i]: int((seq_y == i).sum()) for i in range(4)}
print(f"Prepared {len(seq_X)} causal sequences of shape {seq_X.shape} ({seq_dates[0]} -> {seq_dates[-1]})")
print("Regime label distribution:", counts)

In [ ]:
# Cell 4: PyTorch Causal TCN + Multi-Head Attention Teacher & Distilled Edge MLP Student
class CausalTCNBlock(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3, dilation: int = 1, dropout: float = 0.15):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, dilation=dilation)
        self.norm1 = nn.GroupNorm(4, channels)
        self.norm2 = nn.GroupNorm(4, channels)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Left-only causal padding guarantees zero future leakage inside convolutions
        h = F.pad(x, (self.pad, 0))
        h = self.drop(F.gelu(self.norm1(self.conv1(h))))
        h = F.pad(h, (self.pad, 0))
        h = self.drop(F.gelu(self.norm2(self.conv2(h))))
        return x + h


class FlowRegimeTCNAttn(nn.Module):
    def __init__(self, in_features: int = 12, d_model: int = 64, n_heads: int = 4, dropout: float = 0.18):
        super().__init__()
        self.in_proj = nn.Conv1d(in_features, d_model, kernel_size=1)
        self.tcn1 = CausalTCNBlock(d_model, kernel_size=3, dilation=1, dropout=dropout)
        self.tcn2 = CausalTCNBlock(d_model, kernel_size=3, dilation=2, dropout=dropout)
        self.tcn3 = CausalTCNBlock(d_model, kernel_size=3, dilation=4, dropout=dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm_attn = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )
        self.norm_ffn = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, 6)  # 4 regime logits + 24h return + 72h vol

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        # x_seq: (B, T, F)
        h = self.in_proj(x_seq.transpose(1, 2))
        h = self.tcn3(self.tcn2(self.tcn1(h)))
        h_seq = h.transpose(1, 2)  # (B, T, D)
        t_len = h_seq.size(1)
        causal_mask = torch.triu(
            torch.ones(t_len, t_len, device=h_seq.device, dtype=torch.bool),
            diagonal=1,
        )
        attn_out, _ = self.attn(h_seq, h_seq, h_seq, attn_mask=causal_mask, need_weights=False)
        z = self.norm_attn(h_seq + attn_out)
        z = self.norm_ffn(z + self.ffn(z))
        return self.head(z[:, -1, :])


class DistilledEdgeMLP(nn.Module):
    # Exact 12 -> 16 (tanh) -> 6 student matching Cloudflare Worker `src/regime_engine.mjs`
    def __init__(self, in_dim: int = 12, hidden_dim: int = 16, out_dim: int = 6):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(torch.tanh(self.fc1(x)))


sample_model = FlowRegimeTCNAttn().to(DEVICE)
n_params = sum(p.numel() for p in sample_model.parameters())
print(f"FlowRegimeTCNAttn Teacher initialized: {n_params:,} parameters on {DEVICE}")

In [ ]:
# Cell 5: Strict 5-Fold Purged Expanding Walk-Forward Training & OOS Portfolio Evaluation
EMBARGO_DAYS = 10  # Strictly >= HORIZON_DAYS (7d) to guarantee zero lookahead leakage
N_FOLDS = 5
FEE_BPS = 0.0006   # 6 bps blended SoDEX spot/SSI rebalance friction
MAX_DD_STOP = 0.08
TARGET_VOL = 0.28


def fit_scaler(x_train_seq: np.ndarray):
    flat = x_train_seq.reshape(-1, x_train_seq.shape[-1])
    mean = flat.mean(axis=0)
    std = np.maximum(flat.std(axis=0), 1e-5)
    return mean.astype(np.float32), std.astype(np.float32)


def transform_seq(x_seq: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return np.clip((x_seq - mean) / std, -4.5, 4.5).astype(np.float32)


def train_tcn_fold(x_tr, y_tr, r_tr, v_tr, epochs: int = 45, batch_size: int = 32, lr: float = 1.5e-3):
    model = FlowRegimeTCNAttn().to(DEVICE)
    counts = np.bincount(y_tr, minlength=4).astype(np.float32) + 1.0
    class_w = torch.tensor(np.sqrt(counts.sum() / (4.0 * counts)), dtype=torch.float32, device=DEVICE)

    ds = TensorDataset(
        torch.tensor(x_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.long),
        torch.tensor(r_tr, dtype=torch.float32),
        torch.tensor(v_tr, dtype=torch.float32),
    )
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    use_amp = DEVICE.type == "cuda"
    for _ in range(epochs):
        model.train()
        for bx, by, br, bv in loader:
            bx, by, br, bv = bx.to(DEVICE), by.to(DEVICE), br.to(DEVICE), bv.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=use_amp):
                out = model(bx)
                loss_cls = F.cross_entropy(out[:, :4], by, weight=class_w)
                pred_ret = torch.tanh(out[:, 4]) * 0.08
                loss_ret = F.huber_loss(pred_ret, br.clamp(-0.08, 0.08), delta=0.02) * 20.0
                pred_vol = 0.15 + 1.45 * torch.sigmoid(out[:, 5])
                loss_vol = F.huber_loss(pred_vol, bv.clamp(0.15, 1.60), delta=0.10)
                loss = loss_cls + loss_ret + 0.5 * loss_vol
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        sched.step()
    model.eval()
    return model


def compute_regime_allocation(probs, exp_ret_24h, exp_vol_ann, raw_feats, running_dd):
    daily_vol = max(exp_vol_ann / math.sqrt(365.0), 0.012)
    raw_kelly = exp_ret_24h / (daily_vol * daily_vol)
    frac_kelly = max(0.0, min(0.25, raw_kelly * 0.25))

    p_acc, p_exp, p_chop, p_dd = probs
    bull_prob = p_acc + p_exp
    risk_off_prob = p_dd + 0.65 * p_chop
    btc_ret_7d = float(raw_feats[1])
    btc_ret_30d = float(raw_feats[2])
    flow_z7 = float(raw_feats[8])
    dd_30 = float(raw_feats[10])
    vol_scalar = max(0.15, min(1.0, TARGET_VOL / max(exp_vol_ann, 0.18)))
    trend_score = 0.5 * float(btc_ret_30d > 0) + 0.3 * float(btc_ret_7d > 0) + 0.2 * float(flow_z7 > 0)

    if p_dd >= 0.34 or (btc_ret_30d < -0.04 and flow_z7 < 0.35) or dd_30 < -0.12:
        base_risk = 0.0
    else:
        conviction = max(0.0, bull_prob - risk_off_prob)
        base_risk = max(
            0.0,
            min(0.85, (0.65 * conviction * (0.4 + 0.6 * trend_score) + 1.15 * frac_kelly) * vol_scalar),
        )
        if base_risk < 0.08:
            base_risk = 0.0

    if running_dd >= MAX_DD_STOP:
        base_risk = 0.0
    elif running_dd >= MAX_DD_STOP * 0.5:
        taper = 1.0 - (running_dd - MAX_DD_STOP * 0.5) / (MAX_DD_STOP * 0.5)
        base_risk *= max(0.0, min(1.0, taper))

    btc_w = base_risk * (0.56 if p_exp >= p_acc else 0.64)
    eth_w = base_risk * (0.24 if p_exp >= p_acc else 0.18)
    ssi_w = base_risk * (0.20 if p_exp >= p_acc else 0.18)
    usdc_w = max(0.0, 1.0 - (btc_w + eth_w + ssi_w))
    return {"BTC": btc_w, "ETH": eth_w, "SSI": ssi_w, "USDC": usdc_w}


def perf_metrics(rets):
    arr = np.asarray(rets, dtype=np.float64)
    eq = np.cumprod(1.0 + arr)
    peak = np.maximum.accumulate(np.concatenate([[1.0], eq]))[1:]
    max_dd = float(np.max((peak - eq) / peak))
    final_eq = float(eq[-1])
    cagr = final_eq ** (365.0 / len(arr)) - 1.0
    ann_vol = float(np.std(arr, ddof=1) * np.sqrt(365.0))
    down_std = float(np.sqrt(np.mean(np.minimum(0.0, arr) ** 2)) * np.sqrt(365.0))
    sharpe = float((np.mean(arr) * 365.0) / max(ann_vol, 1e-6))
    sortino = float((np.mean(arr) * 365.0) / max(down_std, 1e-6))
    calmar = float(cagr / max(max_dd, 1e-4))
    hit_rate = float(np.mean(arr > 0.0))
    return {
        "total_return_pct": round((final_eq - 1.0) * 100.0, 2),
        "cagr_pct": round(cagr * 100.0, 2),
        "annualized_vol_pct": round(ann_vol * 100.0, 2),
        "sharpe": round(sharpe, 3),
        "sortino": round(sortino, 3),
        "max_drawdown_pct": round(max_dd * 100.0, 2),
        "calmar": round(calmar, 3),
        "hit_rate_pct": round(hit_rate * 100.0, 2),
        "final_equity": round(final_eq, 4),
    }


n_total = len(seq_X)
min_train = int(n_total * 0.35)
fold_size = (n_total - min_train) // N_FOLDS

oos_strat_rets, oos_bh_rets, oos_dates = [], [], []
oos_strat_curve, oos_bh_curve = [], []
fold_reports = []
cls_hits, cls_total = 0, 0
strat_eq, strat_peak, bh_eq = 1.0, 1.0, 1.0
prev_alloc = {"BTC": 0.0, "ETH": 0.0, "SSI": 0.0, "USDC": 1.0}

for fold in range(N_FOLDS):
    test_start = min_train + fold * fold_size
    test_end = n_total if fold == N_FOLDS - 1 else min_train + (fold + 1) * fold_size
    train_end = test_start - EMBARGO_DAYS

    sc_mean, sc_std = fit_scaler(seq_X[:train_end])
    x_tr = transform_seq(seq_X[:train_end], sc_mean, sc_std)
    x_te = transform_seq(seq_X[test_start:test_end], sc_mean, sc_std)

    model = train_tcn_fold(x_tr, seq_y[:train_end], seq_r1[:train_end], seq_vol[:train_end])
    with torch.no_grad():
        preds = model(torch.tensor(x_te, dtype=torch.float32, device=DEVICE)).cpu().numpy()

    f_strat, f_bh = [], []
    for k in range(len(x_te)):
        global_idx = test_start + k
        logits = preds[k, :4]
        probs = np.exp(logits - logits.max())
        probs = probs / probs.sum()
        if int(np.argmax(probs)) == int(seq_y[global_idx]):
            cls_hits += 1
        cls_total += 1

        exp_ret24 = float(np.tanh(preds[k, 4]) * 0.08)
        exp_vol = float(0.15 + 1.45 / (1.0 + np.exp(-np.clip(preds[k, 5], -10.0, 10.0))))
        running_dd = (strat_peak - strat_eq) / max(strat_peak, 1e-6)
        alloc = compute_regime_allocation(probs, exp_ret24, exp_vol, cur_X[global_idx], running_dd)

        turnover = sum(abs(alloc[a] - prev_alloc[a]) for a in ("BTC", "ETH", "SSI"))
        ssi_ret = (
            0.45 * float(seq_r1[global_idx])
            + 0.30 * float(seq_re1[global_idx])
            + 0.25 * float(seq_rs1[global_idx])
        )
        usdc_carry = 0.042 / 365.0
        port_ret = (
            alloc["BTC"] * float(seq_r1[global_idx])
            + alloc["ETH"] * float(seq_re1[global_idx])
            + alloc["SSI"] * ssi_ret
            + alloc["USDC"] * usdc_carry
            - turnover * FEE_BPS
        )
        bh_ret = float(seq_r1[global_idx])
        strat_eq *= 1.0 + port_ret
        strat_peak = max(strat_peak, strat_eq)
        bh_eq *= 1.0 + bh_ret
        prev_alloc = alloc

        f_strat.append(port_ret)
        f_bh.append(bh_ret)
        oos_strat_rets.append(port_ret)
        oos_bh_rets.append(bh_ret)
        oos_dates.append(seq_dates[global_idx])
        oos_strat_curve.append(round(strat_eq, 4))
        oos_bh_curve.append(round(bh_eq, 4))

    fs = perf_metrics(f_strat)
    fb = perf_metrics(f_bh)
    fold_reports.append(
        {
            "fold": fold + 1,
            "train_range": f"{seq_dates[0]}..{seq_dates[train_end - 1]}",
            "embargo_days": EMBARGO_DAYS,
            "test_range": f"{seq_dates[test_start]}..{seq_dates[test_end - 1]}",
            "test_days": test_end - test_start,
            "strategy_return_pct": fs["total_return_pct"],
            "benchmark_return_pct": fb["total_return_pct"],
            "strategy_sharpe": fs["sharpe"],
            "strategy_max_dd_pct": fs["max_drawdown_pct"],
            "benchmark_max_dd_pct": fb["max_drawdown_pct"],
        }
    )
    print(
        f"Fold {fold + 1}/{N_FOLDS} ({seq_dates[test_start]}..{seq_dates[test_end - 1]}): "
        f"FlowRegime {fs['total_return_pct']:+.2f}% (MaxDD {fs['max_drawdown_pct']:.2f}%) "
        f"vs BTC {fb['total_return_pct']:+.2f}% (MaxDD {fb['max_drawdown_pct']:.2f}%)"
    )

strat_summary = perf_metrics(oos_strat_rets)
strat_summary["regime_classifier_oos_accuracy_pct"] = round(100.0 * cls_hits / max(cls_total, 1), 2)
bh_summary = perf_metrics(oos_bh_rets)
print("\n=== PURGED WALK-FORWARD OOS SCORECARD ===")
print("FlowRegime One (TCN-Attn):", json.dumps(strat_summary, indent=2))
print("BTC Buy & Hold Benchmark: ", json.dumps(bh_summary, indent=2))

In [ ]:
# Cell 6: Visual Walk-Forward Equity Curve & Drawdown Comparison
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
fig.patch.set_facecolor("#060911")
for ax in (ax1, ax2):
    ax.set_facecolor("#0c1220")
    ax.tick_params(colors="#8b9bb8")
    for spine in ax.spines.values():
        spine.set_color("#1d2a44")

x_axis = np.arange(len(oos_dates))
eq_fr = np.asarray(oos_strat_curve)
eq_bh = np.asarray(oos_bh_curve)
dd_fr = (eq_fr / np.maximum.accumulate(eq_fr) - 1.0) * 100.0
dd_bh = (eq_bh / np.maximum.accumulate(eq_bh) - 1.0) * 100.0

ax1.plot(
    x_axis,
    eq_fr,
    color="#10b981",
    linewidth=2.2,
    label=f"FlowRegime One ({strat_summary['total_return_pct']:+.2f}%, Sharpe {strat_summary['sharpe']:.2f})",
)
ax1.plot(
    x_axis,
    eq_bh,
    color="#ef4444",
    linewidth=1.6,
    alpha=0.8,
    label=f"BTC Buy & Hold ({bh_summary['total_return_pct']:+.2f}%, Sharpe {bh_summary['sharpe']:.2f})",
)
ax1.axhline(1.0, color="#2a3d63", linestyle="--", linewidth=1)
ax1.set_title("Purged Walk-Forward Out-of-Sample Equity Curve (10-Day Embargo, Zero Leakage)", color="#e6edf7", fontsize=12)
ax1.set_ylabel("Equity (1.0 = Start)", color="#8b9bb8")
ax1.legend(facecolor="#0c1220", edgecolor="#1d2a44", labelcolor="#e6edf7")

ax2.fill_between(x_axis, dd_bh, 0, color="#ef4444", alpha=0.25, label="BTC Drawdown %")
ax2.fill_between(x_axis, dd_fr, 0, color="#10b981", alpha=0.45, label="FlowRegime Drawdown %")
ax2.axhline(-8.0, color="#f59e0b", linestyle=":", linewidth=1.3, label="Hard 8% Circuit-Breaker")
ax2.set_ylabel("Drawdown (%)", color="#8b9bb8")
ax2.set_xlabel("Out-of-Sample Trading Days", color="#8b9bb8")
ax2.legend(facecolor="#0c1220", edgecolor="#1d2a44", labelcolor="#e6edf7")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 7: Train Full GPU Teacher & Knowledge-Distill to <1ms Edge MLP Student
full_mean, full_std = fit_scaler(seq_X)
x_all_seq = transform_seq(seq_X, full_mean, full_std)
x_all_cur = np.clip((cur_X - full_mean) / full_std, -4.5, 4.5).astype(np.float32)

teacher = train_tcn_fold(x_all_seq, seq_y, seq_r1, seq_vol, epochs=55)
teacher.eval()
with torch.no_grad():
    teacher_out = teacher(torch.tensor(x_all_seq, dtype=torch.float32, device=DEVICE)).cpu()

# Distill teacher soft probabilities + calibrated regression outputs into 12 -> 16 -> 6 Edge MLP
student = DistilledEdgeMLP(12, 16, 6).to(DEVICE)
opt_s = torch.optim.AdamW(student.parameters(), lr=8e-3, weight_decay=1.5e-2)
T_DISTILL = 1.8

x_cur_t = torch.tensor(x_all_cur, dtype=torch.float32, device=DEVICE)
t_out_dev = teacher_out.to(DEVICE)
y_true_t = torch.tensor(seq_y, dtype=torch.long, device=DEVICE)

for _ in range(300):
    student.train()
    opt_s.zero_grad(set_to_none=True)
    s_out = student(x_cur_t)
    p_teacher = F.softmax(t_out_dev[:, :4] / T_DISTILL, dim=-1)
    log_p_student = F.log_softmax(s_out[:, :4] / T_DISTILL, dim=-1)
    loss_kd = F.kl_div(log_p_student, p_teacher, reduction="batchmean") * (T_DISTILL ** 2)
    loss_ce = F.cross_entropy(s_out[:, :4], y_true_t)
    loss_reg = F.mse_loss(s_out[:, 4:], t_out_dev[:, 4:])
    loss = 0.65 * loss_kd + 0.35 * loss_ce + 1.5 * loss_reg
    loss.backward()
    opt_s.step()

student.eval()
with torch.no_grad():
    s_final = student(x_cur_t).cpu()
    t_cls = teacher_out[:, :4].argmax(dim=-1)
    s_cls = s_final[:, :4].argmax(dim=-1)
    fidelity = float((t_cls == s_cls).float().mean().item() * 100.0)

student_cpu = student.cpu()
probe_x = torch.tensor(x_all_cur[-1:], dtype=torch.float32)
t0 = time.perf_counter()
for _ in range(500):
    _ = student_cpu(probe_x)
avg_ms = (time.perf_counter() - t0) * 1000.0 / 500.0
print(f"Distillation complete | Teacher-Student Agreement: {fidelity:.1f}% | Edge CPU Latency: {avg_ms:.4f} ms/req")

In [ ]:
# Cell 8: Export Drop-In Cloudflare Worker Artifacts (model_weights.json & walkforward_metrics.json)
step = max(1, len(oos_dates) // 60)
sampled_curve = [
    {
        "date": oos_dates[i],
        "flowregime_equity": oos_strat_curve[i],
        "btc_bh_equity": oos_bh_curve[i],
    }
    for i in range(0, len(oos_dates), step)
]
if sampled_curve[-1]["date"] != oos_dates[-1]:
    sampled_curve.append(
        {
            "date": oos_dates[-1],
            "flowregime_equity": oos_strat_curve[-1],
            "btc_bh_equity": oos_bh_curve[-1],
        }
    )

w1 = student_cpu.fc1.weight.detach().numpy().tolist()
b1 = student_cpu.fc1.bias.detach().numpy().tolist()
w2 = student_cpu.fc2.weight.detach().numpy().tolist()
b2 = student_cpu.fc2.bias.detach().numpy().tolist()

now_utc = datetime.now(tz=timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

weights_payload = {
    "schema_version": "flowregime-edge-mlp-v1",
    "architecture": "Distilled-TCN-Attn-to-EdgeMLP-12x16x6",
    "trained_at_utc": now_utc,
    "training_window": {
        "start_date": seq_dates[0],
        "end_date": seq_dates[-1],
        "n_samples": int(len(seq_X)),
        "teacher_student_agreement_pct": round(fidelity, 2),
        "data_sources": (
            ["SoSoValue OpenAPI v1 US Spot BTC ETF", "Kraken Daily OHLCV (BTC/ETH/SOL)", "Alternative.me Fear & Greed"]
            if USING_OFFICIAL_SOSO_ETF
            else ["Kraken Daily OHLCV (BTC/ETH/SOL)", "DefiLlama Global Stablecoin Flows", "Alternative.me Fear & Greed"]
        ),
    },
    "regimes": REGIMES,
    "feature_names": FEATURE_NAMES,
    "scaler": {
        "mean": [round(float(v), 6) for v in full_mean],
        "std": [round(float(v), 6) for v in full_std],
        "clip": 4.5,
    },
    "layers": {
        "w1": [[round(float(v), 6) for v in row] for row in w1],
        "b1": [round(float(v), 6) for v in b1],
        "w2": [[round(float(v), 6) for v in row] for row in w2],
        "b2": [round(float(v), 6) for v in b2],
    },
    "output_heads": {
        "regime_logits_indices": [0, 1, 2, 3],
        "exp_return_24h_index": 4,
        "exp_return_24h_scale": 0.08,
        "exp_vol_72h_index": 5,
        "exp_vol_72h_min": 0.15,
        "exp_vol_72h_span": 1.45,
    },
    "risk_defaults": {
        "max_kelly_fraction": 0.25,
        "default_target_vol": 0.28,
        "hard_drawdown_stop": 0.08,
    },
}

metrics_payload = {
    "schema_version": "flowregime-walkforward-audit-v1",
    "generated_at_utc": now_utc,
    "methodology": {
        "validation": "Purged Expanding Walk-Forward Cross-Validation (PyTorch TCN-Attention)",
        "folds": N_FOLDS,
        "embargo_days": EMBARGO_DAYS,
        "lookahead_leakage": False,
        "transaction_cost_bps": round(FEE_BPS * 10000, 1),
        "max_kelly_fraction": 0.25,
        "hard_drawdown_circuit_breaker_pct": round(MAX_DD_STOP * 100, 1),
        "oos_window": f"{oos_dates[0]} -> {oos_dates[-1]}",
        "oos_days": len(oos_dates),
    },
    "flowregime_strategy": strat_summary,
    "btc_buy_and_hold_benchmark": bh_summary,
    "drawdown_reduction_pct": round(bh_summary["max_drawdown_pct"] - strat_summary["max_drawdown_pct"], 2),
    "sharpe_improvement": round(strat_summary["sharpe"] - bh_summary["sharpe"], 3),
    "folds": fold_reports,
    "equity_curve": sampled_curve,
}

out_dir = Path("flowregime_exports")
out_dir.mkdir(exist_ok=True)
(out_dir / "model_weights.json").write_text(json.dumps(weights_payload, indent=2) + "\n", encoding="utf-8")
(out_dir / "walkforward_metrics.json").write_text(json.dumps(metrics_payload, indent=2) + "\n", encoding="utf-8")
print("Exported drop-in Worker artifacts to:", out_dir.resolve())

try:
    from google.colab import files  # type: ignore

    files.download(str(out_dir / "model_weights.json"))
    files.download(str(out_dir / "walkforward_metrics.json"))
except Exception:
    print("Copy flowregime_exports/model_weights.json & walkforward_metrics.json into products/sosovalue_flowregime/src/")